# Tutorial 52: multi-phase training with a `TaskGroup` envelope

Two `@client.task` phases train a ResNet-18 on Tiny ImageNet on **one warm
worker**. Instead of hand-picking a `group_id` and duplicating pins on every
task (tutorial 17), `client.group(phase1, phase2)` derives the shared
envelope from the members themselves: the VRAM floor is the max over members,
pins must not conflict, and the disk envelope covers both members' data — so
the worker provisioned by phase 1 fits the whole group, and phase 2 reuses it
warm together with the downloaded dataset (download ≈ 0 s) and the checkpoint
left in `/data`.

Needs the `tiny-imagenet` data source and an `upload-folder` output source
registered on the broker (see the data-preparation note in
`cas-client/tutorial/52_group_envelope.py`).


In [ ]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

In [ ]:
from krauncher import KrauncherClient

client = KrauncherClient()

INPUT_SOURCE = "tiny-imagenet"
OUTPUT_SOURCE = "upload-folder"

TASK_TIMEOUT = 6000  # seconds — both worker execution and client wait
VRAM_GB = 4
# Phase 2 pins higher on purpose: the group envelope must provision the
# phase-1 worker at the group floor (ceil(17*1.1) = 19 GB) so phase 2 can
# reuse it — the affinity check of this tutorial.
VRAM_GB_PHASE2 = 17


In [ ]:
@client.task(
    vram_gb=VRAM_GB,
#    gpu_arch=GPU_ARCH,
#    gpu_name=GPU_NAME,
    timeout=TASK_TIMEOUT,
    pip=[],  # torch, torchvision are pre-installed in cas-sandbox
    data=INPUT_SOURCE,
    output=OUTPUT_SOURCE,
    stream_stderr=True,  # live progress in the cell output
)
def train_phase1(epochs: int, batch_size: int, lr: float, max_batches: int = 0):
    """Train ResNet-18 on Tiny ImageNet for the first N epochs."""
    import os
    import tarfile
    import time

    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    from torch.utils.data import DataLoader
    from torchvision.datasets import ImageFolder
    from torchvision.models import resnet18

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # ── Unpack dataset if needed ──
    data_root = "/data/tiny-imagenet-200"
    archive = "/data/tiny-imagenet-200.tar.gz"
    if not os.path.isdir(data_root) and os.path.isfile(archive):
        print("Extracting dataset...")
        t0 = time.time()
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall("/data")
        print(f"Extracted in {time.time() - t0:.1f}s")

    # ── Data loaders ──
    transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(64, padding=4),
        T.ToTensor(),
        T.Normalize([0.4802, 0.4481, 0.3975], [0.2770, 0.2691, 0.2821]),
    ])
    train_ds = ImageFolder(os.path.join(data_root, "train"), transform=transform)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=device.type == "cuda",
    )
    n_batches = len(train_loader) if max_batches <= 0 else min(max_batches, len(train_loader))
    print(f"Training samples: {len(train_ds)}, batches/epoch: {n_batches}")

    # ── Model ──
    model = resnet18(num_classes=200).to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4,
    )
    criterion = nn.CrossEntropyLoss()

    # ── Training loop ──
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for i, (images, labels) in enumerate(train_loader, 1):
            if i > n_batches:
                break
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

            if i % 50 == 0 or i == n_batches:
                print(
                    f"Epoch {epoch}/{epochs}  "
                    f"batch {i}/{n_batches}  "
                    f"loss={running_loss / i:.4f}  "
                    f"acc={correct / total:.3f}",
                    flush=True,
                )

        avg_loss = running_loss / n_batches
        accuracy = correct / total
        history.append({"epoch": epoch, "loss": avg_loss, "accuracy": accuracy})
        print(
            f"Epoch {epoch}/{epochs}  "
            f"loss={avg_loss:.4f}  acc={accuracy:.3f}",
            flush=True,
        )

    # ── Save checkpoint to /output (synced to S3) ──
    os.makedirs("/output", exist_ok=True)
    ckpt = {
        "epoch": epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
    }
    ckpt_name = "checkpoint.pt"
    torch.save(ckpt, f"/output/{ckpt_name}")
    print(f"Checkpoint saved to /output/{ckpt_name}")

    # Also save a copy in /data so phase 2 finds it after rename
    torch.save(ckpt, f"/data/{ckpt_name}")
    print(f"Checkpoint copied to /data/{ckpt_name}")

    return {
        "phase": 1,
        "epochs": epochs,
        "final_loss": round(history[-1]["loss"], 4),
        "final_accuracy": round(history[-1]["accuracy"], 4),
    }


In [ ]:
@client.task(
    vram_gb=VRAM_GB_PHASE2,
#    gpu_arch=GPU_ARCH,
#    gpu_name=GPU_NAME,
    timeout=TASK_TIMEOUT,
    pip=[],  # torch, torchvision are pre-installed in cas-sandbox
    data=INPUT_SOURCE,
    output=OUTPUT_SOURCE,
    stream_stderr=True,  # live progress in the cell output
)
def train_phase2(epochs: int, batch_size: int, lr: float, max_batches: int = 0):
    """Continue training ResNet-18 from the checkpoint left by phase 1."""
    import os
    import tarfile
    import time

    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    from torch.utils.data import DataLoader
    from torchvision.datasets import ImageFolder
    from torchvision.models import resnet18

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # ── Check what phase 1 left on disk ──
    data_files = os.listdir("/data")
    print(f"Files in /data: {data_files}")

    data_root = "/data/tiny-imagenet-200"
    archive = "/data/tiny-imagenet-200.tar.gz"

    has_unpacked = os.path.isdir(data_root)
    has_archive = os.path.isfile(archive)
    ckpt_name = "checkpoint.pt"
    has_checkpoint = os.path.isfile(f"/data/{ckpt_name}")

    print(f"Unpacked dataset: {'found' if has_unpacked else 'NOT found'}")
    print(f"Archive:          {'found' if has_archive else 'NOT found'}")
    print(f"Checkpoint:       {'found' if has_checkpoint else 'NOT found'}")

    if has_unpacked:
        print("Dataset already unpacked (data reuse from phase 1)")
    elif has_archive:
        print("Extracting dataset (fallback)...")
        t0 = time.time()
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall("/data")
        print(f"Extracted in {time.time() - t0:.1f}s")
    else:
        raise RuntimeError("No dataset found in /data — phase 1 data not available")

    # ── Data loaders ──
    transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(64, padding=4),
        T.ToTensor(),
        T.Normalize([0.4802, 0.4481, 0.3975], [0.2770, 0.2691, 0.2821]),
    ])
    train_ds = ImageFolder(os.path.join(data_root, "train"), transform=transform)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=device.type == "cuda",
    )

    # ── Model + checkpoint ──
    model = resnet18(num_classes=200).to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4,
    )
    criterion = nn.CrossEntropyLoss()

    # Try to load checkpoint from /data (left by phase 1 via group reuse)
    ckpt_path = f"/data/{ckpt_name}"
    start_epoch = 0
    history = []
    if os.path.isfile(ckpt_path):
        print(f"Loading checkpoint from disk: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"]
        history = ckpt.get("history", [])
        print(f"Resumed from epoch {start_epoch}, previous accuracy: {history[-1]['accuracy']:.3f}")
    else:
        print("WARNING: No checkpoint found on disk, training from scratch")

    # ── Training loop (continue) ──
    for epoch in range(start_epoch + 1, start_epoch + epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        n_batches = len(train_loader) if max_batches <= 0 else min(max_batches, len(train_loader))
        for i, (images, labels) in enumerate(train_loader, 1):
            if i > n_batches:
                break
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

            if i % 50 == 0 or i == n_batches:
                print(
                    f"Epoch {epoch}/{start_epoch + epochs}  "
                    f"batch {i}/{n_batches}  "
                    f"loss={running_loss / i:.4f}  "
                    f"acc={correct / total:.3f}",
                    flush=True,
                )

        avg_loss = running_loss / n_batches
        accuracy = correct / total
        history.append({"epoch": epoch, "loss": avg_loss, "accuracy": accuracy})
        print(
            f"Epoch {epoch}/{start_epoch + epochs}  "
            f"loss={avg_loss:.4f}  acc={accuracy:.3f}",
            flush=True,
        )

    # ── Save final model to /output ──
    os.makedirs("/output", exist_ok=True)
    torch.save({
        "epoch": start_epoch + epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
    }, "/output/model_final.pt")
    print("Final model saved to /output/model_final.pt")

    total_epochs = start_epoch + epochs
    return {
        "phase": 2,
        "resumed_from_epoch": start_epoch,
        "total_epochs": total_epochs,
        "final_loss": round(history[-1]["loss"], 4),
        "final_accuracy": round(history[-1]["accuracy"], 4),
        "initial_accuracy": round(history[0]["accuracy"], 4) if history else None,
    }


In [ ]:
# Build the group envelope from the member tasks: VRAM floor, shared pins
# and a disk envelope sized for both members' data.
group = await client.group(train_phase1, train_phase2)
print(f"Group:           {group.group_id}")
print(f"  VRAM floor:    {group.vram_floor} GB")
print(f"  Disk envelope: {group.disk_gb} GB")


In [ ]:
h1 = await group.submit(train_phase1, epochs=3, batch_size=128, lr=0.01)
r1 = await h1.wait(timeout=TASK_TIMEOUT)

print(f"Phase 1: loss={r1.output['final_loss']}  acc={r1.output['final_accuracy']}")
print(f"Worker: {r1.worker_id}  GPU: {r1.actual_gpu}")
print(f"Download: {r1.download_sec:.2f}s  Time: {r1.execution_time_sec:.2f}s")


In [ ]:
h2 = await group.submit(train_phase2, epochs=3, batch_size=128, lr=0.005)
r2 = await h2.wait(timeout=TASK_TIMEOUT)

print(f"Phase 2: resumed from epoch {r2.output['resumed_from_epoch']}, "
      f"loss={r2.output['final_loss']}  acc={r2.output['final_accuracy']}")
print(f"Worker: {r2.worker_id}  GPU: {r2.actual_gpu}")
print(f"Download: {r2.download_sec:.2f}s  Time: {r2.execution_time_sec:.2f}s")
print()
same = r1.worker_id == r2.worker_id
print(f"Host affinity: {'confirmed' if same else 'FAILED'} ({r1.worker_id})")
print(f"Data reuse:    phase1 download={r1.download_sec:.2f}s, "
      f"phase2 download={r2.download_sec:.2f}s")
print(f"Accuracy:      {r1.output['final_accuracy']} -> {r2.output['final_accuracy']}")
